In [ ]:
import pandas as pd
import dotenv
from sklearn.compose import make_column_transformer
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import MinMaxScaler, OneHotEncoder, LabelBinarizer
from sklearn.pipeline import make_pipeline
from sklearn.svm import SVC
from azure.identity import DefaultAzureCredential
from azure.ai.ml import MLClient
import numpy as np
import matplotlib.pyplot as plt

import os
import dotenv

dotenv.load_dotenv()

In [ ]:
def get_ml_client():
    ml_client = MLClient(
        DefaultAzureCredential(),
        os.getenv("AZURE_SUBSCRIPTION_ID"),
        os.getenv("AZURE_RESOURCE_GROUP"),
        os.getenv("AZURE_ML_WORKSPACE"),
    )
    return ml_client

ml_client = get_ml_client()

In [ ]:
def fetch_data() -> pd.DataFrame:
    data = ml_client.data.get(
        name="f1-data",
        version="3",
    )

    return pd.read_csv(data.path)



def split_data(df: pd.DataFrame, seed=42, test_size=0.2):
    train, test = train_test_split(df, test_size=test_size, random_state=seed, stratify=df["year"])
    return train, test

In [ ]:
data = fetch_data()
print(data.head())

In [ ]:
data.isnull().sum(axis=0)

In [ ]:
# Something weird going on with constructor Marussia in 2015. Let's drop those.
data[data['positionOrder'].isnull()]

In [ ]:
data = data.dropna()

* circuitId = ID per location of the race
* raceId = ID per race weekend
* constructorId = ID per car manufacturer
* driverId = ID per driver
* qualifyId = ID per race + driver combination

In [ ]:
for col in ['raceId', 'qualifyId', 'driverId', 'constructorId', 'circuitId']:
    print(col, len(data[col].unique()))

In [ ]:
data["qualifyingPosition"].hist(bins=np.arange(30))

In [ ]:
data["positionOrder"].hist(bins=np.arange(30))

In [ ]:
plt.figure(figsize=(5,5))
plt.plot([1,25], [1,25],alpha = 0.3, color='k')

plt.scatter(data["qualifyingPosition"], data["positionOrder"], alpha = 0.01)
plt.xlabel("Qualifying position")
plt.ylabel("Race outcome")
plt.show()

In [ ]:
plt.hist(data["positionOrder"] - data["qualifyingPosition"], bins=np.arange(-20, 25),  edgecolor='k')
plt.xlabel("Positions gained compared to qualifying result")
plt.ylabel("Frequency")
plt.show()

In [ ]:
CATEGORICAL_COLS = ["constructorId", "year", "driverId", "circuitId"]
NUMERICAL_COLS = ["year", "qualifyingPosition"]
FEATURE_COLS = ["constructorId", "year", "driverId", "circuitId", "qualifyingPosition"]
BOOL_COLS = ["pointsFinish", "wonRace", "podiumFinish"]
TARGET_COLS = ["positionOrder", "pointsFinish", "wonRace", "podiumFinish"]

In [ ]:
column_transformer = make_column_transformer(
    (OneHotEncoder(min_frequency=3, handle_unknown="infrequent_if_exist"), CATEGORICAL_COLS),
    (MinMaxScaler(), NUMERICAL_COLS),
)

In [ ]:
train, test = split_data(data)

In [ ]:
pipeline = make_pipeline(column_transformer, SVC(gamma='auto'))

In [ ]:
def get_features_labels(df: pd.DataFrame, target_col: str = 'pointsFinish') -> tuple[pd.DataFrame, pd.DataFrame]:
    return df[FEATURE_COLS], df[target_col].astype(int)

In [ ]:
X_train, y_train = get_features_labels(train, target_col="pointsFinish")

In [ ]:
y_train.to_numpy()

In [ ]:
pipeline.fit(X_train, y_train)

In [ ]:
X_test, y_test = get_features_labels(test, target_col="pointsFinish")

In [ ]:
pipeline.score(X_test, y_test)